# 🔗 LangChain Chains

## 📌 What is a Chain?

A **Chain** in LangChain is a sequence of components connected together, where the output of one component becomes the input of the next.

Think of it like an assembly line in a factory.

```text
Prompt
   ↓
LLM
   ↓
Output Parser
   ↓
Final Output
```

Instead of manually calling each component one by one, LangChain allows us to connect them using the `|` operator.

---

# Why do we need Chains?

Without a chain, we would have to write multiple lines of code for every step.

### Without Chain

```text
User Input
      ↓
Create Prompt
      ↓
Call LLM
      ↓
Receive Response
      ↓
Parse Response
```

### With Chain

```python
chain = prompt | llm | parser
```

Everything is executed automatically.

---

# Components Used

## 1. ChatPromptTemplate

A `ChatPromptTemplate` creates prompts dynamically using placeholders.

### Example

```python
prompt_template = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    ("human", "{input}")
])
```

When executed,

```python
prompt_template.invoke({"input": "What is LLM?"})
```

becomes

```text
System:
You are a helpful assistant.

Human:
What is LLM?
```

---

## 2. ChatOpenAI

This component sends the prompt to the language model.

```python
llm_openai = ChatOpenAI(
    model="openai/gpt-oss-20b:free",
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY")
)
```

Its job is simple:

```text
Prompt
      ↓
Language Model
      ↓
AI Response
```

---

## 3. StrOutputParser

LLMs return an AI Message object.

Example:

```python
AIMessage(
    content="LLM stands for Large Language Model..."
)
```

Usually, we only need the text.

`StrOutputParser` extracts only the string.

```text
AIMessage
      ↓
StrOutputParser
      ↓
"LLM stands for Large Language Model..."
```

---

# Sequential Chain

A Sequential Chain executes one step after another.

```python
chain = prompt_template | llm_openai | str_parser
```

Execution Flow

```text
User Input
      ↓
Prompt Template
      ↓
LLM
      ↓
Output Parser
      ↓
String Output
```

Example

```python
chain.invoke({"input": "What is LLM?"})
```

---

# RunnableLambda

Sometimes, the output from one chain is not in the format required by the next chain.

`RunnableLambda` allows us to transform the output.

Example

```python
def dictionary_maker(text):
    return {"text": text}
```

Now,

```text
"Artificial Intelligence"
```

becomes

```python
{
    "text": "Artificial Intelligence"
}
```

which can be passed to another prompt.

---

# Why do we need RunnableLambda?

Suppose the next prompt expects

```python
{content}
```

but the previous chain returns only

```python
"AI is changing the world."
```

The formats don't match.

RunnableLambda converts

```text
String
```

into

```python
{
    "content": "AI is changing the world."
}
```

making it compatible with the next chain.

---

# Chain with Custom Runnable

```python
prompt
    ↓
LLM
    ↓
Parser
    ↓
RunnableLambda
```

Flow

```text
Question
      ↓
Generate AI Response
      ↓
Convert into Dictionary
      ↓
Return Dictionary
```

---

# Multi-Step Sequential Chain

A chain can contain multiple prompts.

Example:

Step 1

Generate detailed content.

↓

Step 2

Use that content to generate a LinkedIn post.

Flow

```text
Topic
   ↓
Prompt 1
   ↓
LLM
   ↓
Parser
   ↓
Dictionary
   ↓
Prompt 2
   ↓
LLM
   ↓
Parser
   ↓
LinkedIn Post
```

Example

Input

```text
Agentic AI
```

Output

```text
3 Paragraphs about Agentic AI

↓

Professional LinkedIn Post
```

This demonstrates how one AI-generated result can become the input for another AI task.

---

# RunnableParallel

`RunnableParallel` executes multiple chains simultaneously.

Instead of generating only a LinkedIn post, we can generate:

* LinkedIn Post
* Instagram Caption

at the same time.

Example

```python
RunnableParallel(
    branches={
        "linkedin": linkedin_chain,
        "instagram": insta_chain
    }
)
```

Execution

```text
Generated Content
        │
        ├──────────────► LinkedIn Chain
        │
        └──────────────► Instagram Chain
```

Both chains receive the same input and run independently.

---

# Sequential vs Parallel Chain

| Sequential Chain                             | Parallel Chain                          |
| -------------------------------------------- | --------------------------------------- |
| Executes one step after another              | Executes multiple chains simultaneously |
| Output of one step becomes input to the next | Same input is sent to multiple chains   |
| Used for workflows                           | Used for generating multiple outputs    |

---

# Real-Life Example

Imagine writing a blog.

### Sequential

```text
Topic

↓

Generate Article

↓

Generate Summary

↓

Generate LinkedIn Post
```

Each step depends on the previous one.

---

### Parallel

```text
Article

├── LinkedIn Post

├── Instagram Caption

├── Twitter Post

└── Email Newsletter
```

The same article is reused to generate content for multiple platforms simultaneously.

---

# Key Takeaways

* A **Chain** connects multiple LangChain components together.
* `ChatPromptTemplate` creates dynamic prompts.
* `ChatOpenAI` sends prompts to the LLM.
* `StrOutputParser` extracts plain text from the model response.
* `RunnableLambda` transforms data into the format required by the next component.
* **Sequential Chains** execute tasks one after another.
* **RunnableParallel** executes multiple chains simultaneously using the same input.
* Chains help build modular, reusable, and scalable LLM applications.

---

# Interview Questions

### What is a Chain in LangChain?

A Chain is a sequence of connected components where the output of one component becomes the input of the next.

---

### Why do we use StrOutputParser?

It converts the AIMessage returned by the LLM into a plain Python string.

---

### What is RunnableLambda?

RunnableLambda is used to transform or manipulate data between chain components, making outputs compatible with subsequent steps.

---

### What is the difference between Sequential Chain and Parallel Chain?

Sequential Chains execute tasks one after another, while Parallel Chains execute multiple independent chains simultaneously on the same input.

---

### When should RunnableParallel be used?

Use RunnableParallel when multiple independent outputs (such as LinkedIn posts, Instagram captions, summaries, or translations) need to be generated from the same input at the same time.


In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

if os.getenv("OPENROUTER_API_KEY") is not None:
    print("OpenRouter API Key Found")
else:
    print("OpenRouter API Key Not Found")
from langchain_openai import ChatOpenAI

llm_openai = ChatOpenAI(base_url = "https://openrouter.ai/api/v1", model ="openai/gpt-oss-20b:free", temperature = 0.7, api_key = os.getenv("OPENROUTER_API_KEY"))
## **Sequential Chain**
# Task 1

from langchain_core.prompts import ChatPromptTemplate

prompt_template = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    ("human", "{input}")
])
new_prompt = prompt_template.invoke({"input": "What is LLM?"})
new_prompt.messages
# Task 2

response = llm_openai.invoke(new_prompt.messages)
# Task 3 
response.content
from langchain_core.output_parsers import StrOutputParser

# Task 3 - String Parser
str_parser = StrOutputParser()

chain = prompt_template | llm_openai | str_parser
chain.invoke({"input": "What is LLM?"})
## **Chain with Custom Runnable**
# First Create the prompt

# Task 1 - Prompt

prompt_template = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    ("human", "{input}")
])
# Task 2 - LLM 

llm_openai = ChatOpenAI(base_url = "https://openrouter.ai/api/v1", model ="openai/gpt-oss-20b:free", temperature = 0.7, api_key = os.getenv("OPENROUTER_API_KEY"))
# Task 3 - String Parser
str_parser = StrOutputParser()
chain = prompt_template | llm_openai | str_parser
response = chain.invoke({"input": "What is AI ?"})
print(response)
# Task 4 - Custom Runnable
from langchain_core.runnables import RunnableLambda

def dictionary_maker(text:str) -> dict:
    return {"text": text}

dictionary_maker_runnable = RunnableLambda(dictionary_maker)
dictionary_maker_runnable.invoke("Hello")
chain2 = prompt_template | llm_openai | str_parser | dictionary_maker_runnable
chain2.invoke({"input": "what is Machine Learning?"})
## **Task with Chains**
# Task 1

prompt_template = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Generate 3 para content for the given topic"),
    ("human", "Generate 3 para content for the topic: {input}")
])

# Task 2 - LLM 

llm_openai = ChatOpenAI(base_url = "https://openrouter.ai/api/v1", model ="openai/gpt-oss-20b:free", temperature = 0.7, api_key = os.getenv("OPENROUTER_API_KEY"))
# Task 3 - String Parser

str_parser = StrOutputParser()
# Task 4 - Custom Runnable
from langchain_core.runnables import RunnableLambda

def dictionary_maker(text:str) -> dict:
    return {"content": text}

dictionary_maker_runnable = RunnableLambda(dictionary_maker)
# {"content": "AI is about......"}
# Task 5

prompt_template_2 = ChatPromptTemplate.from_messages([
    ("system", "You are a professional Linked In Post Generator"),
    ("human", "Generate a linked post for following content: {content}")
])
# Task 6

llm_openai = ChatOpenAI(base_url = "https://openrouter.ai/api/v1", model ="openai/gpt-oss-20b:free", temperature = 0.7, api_key = os.getenv("OPENROUTER_API_KEY"))
# Task 7 - String Parser

str_parser = StrOutputParser()
final_chain = (prompt_template | 
                llm_openai | 
                str_parser | 
                dictionary_maker_runnable | 
                prompt_template_2 | 
                llm_openai | 
                str_parser
            )
final_chain.invoke({"input": "Agentic AI"})

## **Parallel Chain**
# Task 1

prompt_template = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Generate 3 para content for the given topic"),
    ("human", "Generate 3 para content for the topic: {input}")
])

# Task 2 - LLM 

llm_openai = ChatOpenAI(base_url = "https://openrouter.ai/api/v1", model ="openai/gpt-oss-20b:free", temperature = 0.7, api_key = os.getenv("OPENROUTER_API_KEY"))
# Task 3 - String Parser

str_parser = StrOutputParser()
# Task 4 - Custom Runnable
from langchain_core.runnables import RunnableLambda

def dictionary_maker(text:str) -> dict:
    return {"content": text}

dictionary_maker_runnable = RunnableLambda(dictionary_maker)
# {"content":"................."}
prompt_template_linkedin = ChatPromptTemplate.from_messages([
    ("system", "You are a professional Linked In Post Generator"),
    ("human", "Generate a linked post for following content: {content}")
])

prompt_template_instagram = ChatPromptTemplate.from_messages([
    ("system", "You are a professional Instagram Post Generator"),
    ("human", "Generate a Instagram post for following content: {content}")
])

linkedin_chain = prompt_template_linkedin | llm_openai | str_parser
insta_chain = prompt_template_instagram | llm_openai | str_parser
from langchain_core.runnables import RunnableParallel

final_chain = (prompt_template | 
        llm_openai | 
        str_parser | 
        dictionary_maker_runnable | 
        RunnableParallel(branches = {"linkedin": linkedin_chain, "instagram": insta_chain})
        )
response = final_chain.invoke({"input":"Agentic AI"})
response
print(response["branches"]["linkedin"])
print(response["branches"]["instagram"])




OpenRouter API Key Found
**Artificial Intelligence (AI)** is a field of computer science that focuses on creating systems capable of performing tasks that normally require human intelligence. These tasks include:

| Task | Typical श… | AI Approach |
|------|-----------|-------------|
| Recognizing speech | Voice‑to‑text | Speech‑recognition neural nets |
| Interpreting images | Object detection | Convolutional neural networks |
| Understanding language | Sentiment analysis | Transformer models |
| Making decisions | Game‑playing, stock trading | Reinforcement learning |
| Planning & reasoning | Robot navigation | Symbolic AI + planning algorithms |

### Core ideas behind AI

1. **Representation** – How to encode knowledge or data (e.g., numbers, graphs, logical rules).
2. **Learning** – Algorithms that improve from data (supervised, unsupervised, reinforcement).
3. **Reasoning** – Drawing conclusions, solving problems, planning.
4. **Perception** – Interpreting raw signals (vision, aud